# Create evaluation notebook to write the thresholds and detection result files.

In [1]:
# Import Functions
import numpy as np
import polars as pl

# Read parquet files
ridge = pl.read_parquet("../results/scores_ridge.parquet")
pca = pl.read_parquet("../results/scores_pca.parquet")

# Set up global variables
RUN = "simulationRun"
FAULT = "faultNumber"


### 1. **Thresholds**. For each of the three statistics (T2, SPE, ridge), the threshold is numpy.quantile(scores, 0.99) over the validation runs, with NumPy’s default interpolation. Write results/thresholds.csv with columns detector, threshold.

#### Take all the validation scores for each stat and take the 99th percentile. Then compute the thresholds for each stat.

In [2]:
# Compute the thresholds for each statistic

ridge_filtered = ridge.filter((pl.col(RUN) >= 301) & (pl.col(RUN) <= 400) & (pl.col(FAULT) == 0))
# get the ridge score column and convert it to a numpy array 
ridge_score = ridge_filtered.get_column("score") 
scores = ridge_score.to_numpy()
thresh_ridge = np.quantile(scores, 0.99)


pca_filtered = pca.filter((pl.col(RUN) >= 301) & (pl.col(RUN) <= 400) & (pl.col(FAULT) == 0))
# Get the T2 and SPE score columns and convert them to numpy arrays
T2_score = pca_filtered.get_column("T2") 
T2_score = T2_score.to_numpy() 
thresh_T2 = np.quantile(T2_score, 0.99)

SPE_score = pca_filtered.get_column("SPE")
SPE_score = SPE_score.to_numpy()
thresh_SPE = np.quantile(SPE_score, 0.99)

# Create threshold result table
data = {"detector": ["T2", "SPE", "Ridge"], "threshold": [thresh_T2, thresh_SPE, thresh_ridge]}
thresholds = pl.DataFrame(data)
thresholds

# Write it to results
thresholds.write_csv("../results/thresholds.csv")

### 2. **Alarms**. A sample is in alarm when it and the two samples before it all exceed the threshold. Alarms never cross a run boundary.

In [3]:
def above_threshold_check(df, threshold, score): # score is "score", "T2", "SPE"
    # Go through each element in the scores table to check if score is above threshold
    new_df = df.with_columns(
        pl.when(pl.col(score) > threshold).then(True).otherwise(False).alias("above_threshold") )  
    # Sort the dataframe
    new_df = new_df.sort([FAULT, RUN, "sample"])
    # Check if it and the two samples before exceed the threshold
    check = new_df.with_columns(
        pl.col("above_threshold").shift(1).over(FAULT, RUN).alias("one_sample_above"),
        pl.col("above_threshold").shift(2).over(FAULT, RUN).alias("two_samples_above"))
    check = check.with_columns(
        pl.when((pl.col("above_threshold") == True) & (pl.col("one_sample_above") == True)
                & (pl.col("two_samples_above") == True)).then(True).otherwise(False).alias("in_alarm"))
    # Save the in_alarm results
    alarms = check.select(
        pl.col(FAULT), pl.col(RUN), pl.col("sample"), pl.col("in_alarm")
    )
    # Add a detector column to identify the statistic
    if score == "score":
        alarms = alarms.with_columns(pl.lit("ridge").alias("detector")) 
    else:
        alarms = alarms.with_columns(pl.lit(score).alias("detector"))
        
    return alarms
   

#### Create an alarms table for each statistic and concatenate them together to have one large dataframe

In [4]:
# Create "alarms" table
ridge_alarms = above_threshold_check(ridge, thresh_ridge, "score")

T2_alarms = above_threshold_check(pca, thresh_T2, "T2")

SPE_alarms = above_threshold_check(pca, thresh_SPE, "SPE")

# merge the tables together
alarms = pl.concat([ridge_alarms, T2_alarms, SPE_alarms])
print(alarms)

shape: (898_800, 5)
┌─────────────┬───────────────┬────────┬──────────┬──────────┐
│ faultNumber ┆ simulationRun ┆ sample ┆ in_alarm ┆ detector │
│ ---         ┆ ---           ┆ ---    ┆ ---      ┆ ---      │
│ i64         ┆ i64           ┆ i64    ┆ bool     ┆ str      │
╞═════════════╪═══════════════╪════════╪══════════╪══════════╡
│ 0           ┆ 301           ┆ 3      ┆ false    ┆ ridge    │
│ 0           ┆ 301           ┆ 4      ┆ false    ┆ ridge    │
│ 0           ┆ 301           ┆ 5      ┆ false    ┆ ridge    │
│ 0           ┆ 301           ┆ 6      ┆ false    ┆ ridge    │
│ 0           ┆ 301           ┆ 7      ┆ false    ┆ ridge    │
│ …           ┆ …             ┆ …      ┆ …        ┆ …        │
│ 20          ┆ 20            ┆ 496    ┆ true     ┆ SPE      │
│ 20          ┆ 20            ┆ 497    ┆ true     ┆ SPE      │
│ 20          ┆ 20            ┆ 498    ┆ true     ┆ SPE      │
│ 20          ┆ 20            ┆ 499    ┆ true     ┆ SPE      │
│ 20          ┆ 20            ┆ 500

### 3. **Metrics, for each detector:**

### False-alarm rate: the share of samples in alarm over the test runs, reported as fault 0;

### Detection rate: for each faulty run, the share of samples after sample 20 that are in alarm, averaged over the 20 runs;

### Detection delay: for each faulty run, minutes from sample 20 to the first alarm after it; report the median over the runs that alarmed, and how many runs never alarmed.

#### 3.1. Compute false alarm rate per detector. This will be reported as fault 0

In [5]:
def false_alarm_rate(df): #takes in combined dataframe from the previous step
    # take in in_alarm dataframe and filter over test runs
    df = df.filter((pl.col(RUN) >= 401) & (pl.col(RUN) <= 500) & (pl.col(FAULT) == 0))
    share = df.select(pl.col("detector"), pl.col(FAULT), pl.col("in_alarm"))
    
    # group by detector when I get the other stats
    share = share.group_by("detector", FAULT).agg(pl.col("in_alarm").mean())
    return share


In [6]:
# Create false alarm rate table
false_rate = false_alarm_rate(alarms)
print(false_rate)

shape: (3, 3)
┌──────────┬─────────────┬──────────┐
│ detector ┆ faultNumber ┆ in_alarm │
│ ---      ┆ ---         ┆ ---      │
│ str      ┆ i64         ┆ f64      │
╞══════════╪═════════════╪══════════╡
│ SPE      ┆ 0           ┆ 0.0      │
│ ridge    ┆ 0           ┆ 0.0      │
│ T2       ┆ 0           ┆ 0.00052  │
└──────────┴─────────────┴──────────┘


#### 3.2. Compute the detection rate

In [7]:
# Through faults 1-20, find the average number of samples after sample 20 that are in alarm.
# This is averaged over 20 runs
def detection_rate(df): # take combined in_alarms dataframe
    df = df.filter((pl.col(FAULT) != 0) & (pl.col("sample") > 20) )
    # Get average number of samples after 20 that are in alarm
    detect = df.group_by("detector", FAULT, RUN).agg(pl.col("in_alarm").mean())
    # average over 20 runs
    detect = detect.group_by("detector", FAULT).agg(pl.col("in_alarm").mean())
    detect = detect.sort(["detector", FAULT])
    detect = detect.select(pl.col("detector"), pl.col(FAULT), pl.col("in_alarm").alias("detection_rate"))
    return detect

    

In [8]:
# Get the detection rate table
d_rate = (detection_rate(alarms))
print(d_rate)

shape: (60, 3)
┌──────────┬─────────────┬────────────────┐
│ detector ┆ faultNumber ┆ detection_rate │
│ ---      ┆ ---         ┆ ---            │
│ str      ┆ i64         ┆ f64            │
╞══════════╪═════════════╪════════════════╡
│ SPE      ┆ 1           ┆ 0.991875       │
│ SPE      ┆ 2           ┆ 0.958958       │
│ SPE      ┆ 3           ┆ 0.0            │
│ SPE      ┆ 4           ┆ 0.994167       │
│ SPE      ┆ 5           ┆ 0.125417       │
│ …        ┆ …           ┆ …              │
│ ridge    ┆ 16          ┆ 0.711875       │
│ ridge    ┆ 17          ┆ 0.866875       │
│ ridge    ┆ 18          ┆ 0.892708       │
│ ridge    ┆ 19          ┆ 0.029167       │
│ ridge    ┆ 20          ┆ 0.635104       │
└──────────┴─────────────┴────────────────┘


#### 3.3. Compute the detection delay per detector

In [9]:
# Compute the detection delay per detector

def detection_delay(df):
    # Filter to faulty runs
    df = df.filter((pl.col(FAULT) != 0) & (pl.col("sample") > 20) & (pl.col("in_alarm") == True))
   
    # Minutes from sample 20 to first alarm after it
    new_df = df.group_by("detector", FAULT, RUN).agg(pl.col("sample").min())
    new_df = new_df.sort(["detector", FAULT, RUN, "sample"])
    new_df = new_df.with_columns(((pl.col("sample")-20)*3).alias("minutes_from_20"))

    # Find how many runs never alarmed
    count = new_df.group_by("detector", FAULT).agg(pl.col(RUN).count()) #number of runs that have alarmed per fault
    count = count.sort(["detector", FAULT])
    never_alarmed = count.select(pl.col(RUN)-20) # create a new column showing runs that never alarmed
    
    # Report the median over the runs that alarmed
    delay = new_df.group_by("detector", FAULT).agg(pl.col("minutes_from_20").median())
    delay = delay.sort(["detector", FAULT])
    delay = delay.with_columns(never_alarmed)
    metrics = delay.select(pl.col("detector"), pl.col(FAULT),
        (pl.col("minutes_from_20").alias("median_delay_min")), (pl.col(RUN).alias("runs_missed")))

    return metrics 


In [10]:
# Print it
d_delay = detection_delay(alarms)
print(d_delay)

shape: (54, 4)
┌──────────┬─────────────┬──────────────────┬─────────────┐
│ detector ┆ faultNumber ┆ median_delay_min ┆ runs_missed │
│ ---      ┆ ---         ┆ ---              ┆ ---         │
│ str      ┆ i64         ┆ f64              ┆ u32         │
╞══════════╪═════════════╪══════════════════╪═════════════╡
│ SPE      ┆ 1           ┆ 15.0             ┆ 0           │
│ SPE      ┆ 2           ┆ 57.0             ┆ 0           │
│ SPE      ┆ 4           ┆ 12.0             ┆ 0           │
│ SPE      ┆ 5           ┆ 15.0             ┆ 0           │
│ SPE      ┆ 6           ┆ 9.0              ┆ 0           │
│ …        ┆ …           ┆ …                ┆ …           │
│ ridge    ┆ 16          ┆ 45.0             ┆ 0           │
│ ridge    ┆ 17          ┆ 118.5            ┆ 0           │
│ ridge    ┆ 18          ┆ 141.0            ┆ 0           │
│ ridge    ┆ 19          ┆ 121.5            ┆ 0           │
│ ridge    ┆ 20          ┆ 151.5            ┆ 0           │
└──────────┴─────────────

### Write results/detection.csv with columns fault, detector, detection_rate, median_delay_min, runs_missed. (21 rows for each detector, faults 0 to 20)

In [11]:
# Join the false alarm rate, detection rate, and detection delay tables together
detection = d_rate.join(d_delay, on=["detector", FAULT], how="left") # join the detection rate and delay tables
# add the false alarm rates
f = false_rate.select(pl.col("detector"), pl.col(FAULT), pl.col("in_alarm").alias("detection_rate"))
detection = pl.concat([detection, f], how="diagonal")
detection = detection.sort(["detector", FAULT]) # Sort the detection table by detector and fault
print(detection)

# Write it to csv file
detection.write_csv("../results/detection.csv")


shape: (63, 5)
┌──────────┬─────────────┬────────────────┬──────────────────┬─────────────┐
│ detector ┆ faultNumber ┆ detection_rate ┆ median_delay_min ┆ runs_missed │
│ ---      ┆ ---         ┆ ---            ┆ ---              ┆ ---         │
│ str      ┆ i64         ┆ f64            ┆ f64              ┆ u32         │
╞══════════╪═════════════╪════════════════╪══════════════════╪═════════════╡
│ SPE      ┆ 0           ┆ 0.0            ┆ null             ┆ null        │
│ SPE      ┆ 1           ┆ 0.991875       ┆ 15.0             ┆ 0           │
│ SPE      ┆ 2           ┆ 0.958958       ┆ 57.0             ┆ 0           │
│ SPE      ┆ 3           ┆ 0.0            ┆ null             ┆ null        │
│ SPE      ┆ 4           ┆ 0.994167       ┆ 12.0             ┆ 0           │
│ …        ┆ …           ┆ …              ┆ …                ┆ …           │
│ ridge    ┆ 16          ┆ 0.711875       ┆ 45.0             ┆ 0           │
│ ridge    ┆ 17          ┆ 0.866875       ┆ 118.5            

3.2.1. Check to see if the three faults show up. 
It should be noted that we expect faults 3, 9, and 15 to have zero detection rate because they were not able to be detected according to the provided literature. 

In [12]:
detection.filter((pl.col(FAULT) == 3) | (pl.col(FAULT) == 9) | (pl.col(FAULT) == 15) )

# Note: T2 for the three faults 3, 9, and 15 have a small detection rate but not enough to be detected. Majority of the
# runs are missed and not alarms.

detector,faultNumber,detection_rate,median_delay_min,runs_missed
str,i64,f64,f64,u32
"""SPE""",3,0.0,null,null
"""SPE""",9,0.0,null,null
"""SPE""",15,0.0,null,null
"""T2""",3,0.000208,231.0,4294967277
"""T2""",9,0.000104,231.0,4294967277
"""T2""",15,0.000208,231.0,4294967277
"""ridge""",3,0.0,null,null
"""ridge""",9,0.0,null,null
"""ridge""",15,0.0,null,null
